In [1]:
# 补listen和save
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft, istft
from scipy.io import wavfile
from IPython.display import Audio, display

def save(name, sig, sr=16000):
    sig = np.clip(sig, -1, 0.99997)
    wavfile.write(name, sr, (sig * 32768).astype(np.int16))

def listen(sig, name=None, sr=16000):
    if name:
        save(name, sig, sr)
    display(Audio(np.clip(sig, -1, 0.99997), rate=sr))

print("工具函数就绪")

工具函数就绪


In [2]:
import numpy as np
from scipy.signal import stft, istft

SR, NPERSEG, NOVERLAP = 16000, 400, 240

def synth_speech(dur, f0, seed):
    rng = np.random.default_rng(seed)
    t = np.arange(0, dur, 1/SR)
    kmax = min(14, int(7500/f0))                      # 防止超奈奎斯特
    x = np.zeros(len(t))
    for k in range(1, kmax+1):
        x = x + np.sin(2*np.pi*f0*k*t + rng.uniform(0, 2*np.pi))/k
    x = x * np.sin(2*np.pi*rng.uniform(1.2, 3.5)*t)**2      # 随机音节速率
    for _ in range(rng.integers(1, 4)):                     # 随机插入静音段
        s = rng.uniform(0, dur-0.4)
        x[(t > s) & (t < s + rng.uniform(0.2, 0.5))] = 0
    p = np.max(np.abs(x))
    return x/p*0.5 if p > 0 else x

def colored_noise(beta, n, seed):
    rng = np.random.default_rng(seed)
    s = rng.standard_normal(n//2+1) + 1j*rng.standard_normal(n//2+1)
    fq = np.arange(len(s)).astype(float); fq[0] = 1
    y = np.fft.irfft(s / fq**(beta/2), n=n)
    return y / (np.std(y) + 1e-9)

def make_pair(seed, dur=3.0):
    rng = np.random.default_rng(seed)
    x  = synth_speech(dur, rng.uniform(90, 260), seed)
    nz = colored_noise(rng.choice([0.0, 1.0, 2.0]), len(x), seed+9999)   # 白/粉/布朗
    snr_db = rng.uniform(-5, 15)
    nz = nz * np.sqrt(np.mean(x**2)/(np.mean(nz**2)*10**(snr_db/10)) + 1e-20)
    if rng.random() < 0.5:                                  # 一半样本噪声随时间变
        t = np.arange(len(x))/SR
        nz = nz * (1 + rng.uniform(1,3)/(1+np.exp(-(t-rng.uniform(.5,dur-.5))*4)))
    return x, nz, x+nz

def feats(seed, dur=3.0):
    x, nz, noisy = make_pair(seed, dur)
    S = lambda s: stft(s, SR, nperseg=NPERSEG, noverlap=NOVERLAP)[2]
    Xn, C, N = S(noisy), S(x), S(nz)
    logmag = np.log(np.abs(Xn) + 1e-6).T                    # 输入特征 (帧, 201)
    irm    = np.sqrt(np.abs(C)**2/(np.abs(C)**2+np.abs(N)**2+1e-12)).T   # 答案
    return logmag.astype(np.float32), irm.astype(np.float32)

F, M = feats(0)
print("特征", F.shape, " 掩蔽", M.shape)      # (301, 201) (301, 201)

特征 (301, 201)  掩蔽 (301, 201)


In [3]:
import torch, torch.nn as nn

CTX = 2                                    # 前后各看2帧，共5帧

def stack_ctx(F):
    pad = np.pad(F, ((CTX, CTX), (0, 0)), mode="edge")
    return np.concatenate([pad[i:i+len(F)] for i in range(2*CTX+1)], axis=1)

def build(n, off=0):
    Xs, Ys = [], []
    for i in range(n):
        F, M = feats(off+i)
        Xs.append(stack_ctx(F)); Ys.append(M)
    return torch.from_numpy(np.concatenate(Xs)), torch.from_numpy(np.concatenate(Ys))

Xtr, Ytr = build(120, 0)        # 训练：120条
Xva, Yva = build(20, 5000)      # 验证：20条，seed完全不同

mu, sd = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True) + 1e-6
Xtr, Xva = (Xtr-mu)/sd, (Xva-mu)/sd            # 归一化

print("训练集", tuple(Xtr.shape), " 验证集", tuple(Xva.shape))

训练集 (36120, 1005)  验证集 (6020, 1005)


In [4]:
model = nn.Sequential(
    nn.Linear(201*(2*CTX+1), 512), nn.ReLU(),
    nn.Linear(512, 512),           nn.ReLU(),
    nn.Linear(512, 201),           nn.Sigmoid(),
)
print("参数量", sum(p.numel() for p in model.parameters()))    # 880841

参数量 880841


In [5]:
opt   = torch.optim.Adam(model.parameters(), 1e-3)
lossf = nn.MSELoss()
dl    = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(Xtr, Ytr), batch_size=256, shuffle=True)

for ep in range(12):
    model.train(); tot = 0
    for xb, yb in dl:
        opt.zero_grad()              # ① 清空上一轮的梯度
        loss = lossf(model(xb), yb)  # ② 前向：预测 + 算误差
        loss.backward()              # ③ 反向：算每个参数该往哪调
        opt.step()                   # ④ 更新参数
        tot += loss.item()*len(xb)
    model.eval()
    with torch.no_grad():
        v = lossf(model(Xva), Yva).item()
    print(f"epoch {ep+1:2d}  train {tot/len(Xtr):.5f}  val {v:.5f}")

epoch  1  train 0.03469  val 0.02030
epoch  2  train 0.01564  val 0.01384
epoch  3  train 0.01086  val 0.01216
epoch  4  train 0.00937  val 0.01152
epoch  5  train 0.00862  val 0.01086
epoch  6  train 0.00796  val 0.01034
epoch  7  train 0.00737  val 0.00991
epoch  8  train 0.00719  val 0.01001
epoch  9  train 0.00674  val 0.00953
epoch 10  train 0.00659  val 0.00952
epoch 11  train 0.00656  val 0.00986
epoch 12  train 0.00629  val 0.00938


In [8]:
def enhance(noisy):
    F = np.log(np.abs(stft(noisy, SR, nperseg=NPERSEG, noverlap=NOVERLAP)[2]) + 1e-6).T
    X = torch.from_numpy(stack_ctx(F).astype(np.float32))
    X = (X - mu) / sd                                  # 必须用训练时的 mu/sd
    with torch.no_grad():
        G = model(X).numpy().T                         # 转回 (201, 帧)
    Xn = stft(noisy, SR, nperseg=NPERSEG, noverlap=NOVERLAP)[2]
    return istft(Xn * G, SR, nperseg=NPERSEG, noverlap=NOVERLAP)[1], G

x, nz, noisy = make_pair(9000)          # 一条没见过的
y_nn, G_nn = enhance(noisy)

listen(noisy)
listen(y_nn[:len(x)])